
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lecture - Tool Integration and Observability
#### Duration: 20 minutes

## Overview
This lecture covers how deployed agents connect to external tools through the Model Context Protocol (MCP) and how MLflow Tracing provides the observability foundation for production monitoring. Together, these capabilities make agents extensible and observable — the prerequisites for evaluating and monitoring them in production.

## Learning Objectives
By the end of this lecture you will be able to:
- Describe how MCP enables runtime tool discovery and invocation for deployed agents
- Differentiate managed, external, and custom MCP servers on Databricks
- Explain how an MCP client lists and calls a server's tools
- Identify what MLflow Tracing captures and how it connects deployment to evaluation

## A. Model Context Protocol

### A1. What Is MCP?

The **Model Context Protocol (MCP)** is an open protocol for connecting AI applications to external systems. Think of it like a USB-C port for AI: just as USB-C provides a standardized physical interface for connecting peripherals, MCP provides a standardized software interface that lets AI applications communicate with data sources, tools, and workflows.

Without MCP, every tool integration requires custom code: API wrappers, authentication handling, response parsing. MCP eliminates this by defining a common protocol that any compliant client and server can speak.

MCP follows a **client-server architecture** with three participants:

<div style="background: #fff; max-width: 1600px; margin: 0 auto; padding: 12px 0 24px;">
<svg viewBox="0 0 1600 420" width="100%" role="img" aria-label="Generic MCP architecture. An MCP host, the AI application, runs an agent and one MCP client per server. Each client keeps its own two-way MCP connection to one server, here GitHub and Postgres, and each server exposes tools, resources, and prompts." style="display: block; height: auto; font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;">
<defs>
<marker id="mg-ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
<filter id="mg-sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="3" stdDeviation="6" flood-color="#1B3139" flood-opacity="0.08"/></filter>
</defs>
<defs><marker id="mg-as" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker></defs>
<rect x="40" y="20" width="740" height="376" rx="14" fill="#F9F7F4" stroke="#EEEDE9" stroke-width="1.5"/>
<rect x="62" y="42" width="97" height="26" rx="13.0" fill="#FF5F46"/><text x="110" y="60" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">MCP HOST</text>
<text x="62" y="106" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">AI application</text>
<rect x="62" y="190.0" width="250" height="120" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#mg-sh)"/><text x="187" y="257" font-size="21" font-weight="700" fill="#0b2026" text-anchor="middle" xml:space="preserve">Agent</text>
<rect x="420" y="129.0" width="338" height="92" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#mg-sh)"/><text x="589" y="182" font-size="21" font-weight="700" fill="#0b2026" text-anchor="middle" xml:space="preserve">MCP client</text>
<rect x="420" y="279.0" width="338" height="92" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#mg-sh)"/><text x="589" y="332" font-size="21" font-weight="700" fill="#0b2026" text-anchor="middle" xml:space="preserve">MCP client</text>
<path d="M318,250.0 H380" fill="none" stroke="#618794" stroke-width="2.4"/>
<path d="M380,250.0 V175 H412" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#mg-ar)"/>
<path d="M380,250.0 V325 H412" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#mg-ar)"/>
<rect x="880" y="134" width="520" height="82" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5"/>
<text x="900" y="181" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">GitHub</text>
<rect x="1079" y="158" width="76" height="34" rx="17.0" fill="#F0F3F6" stroke="#C6DCEA" stroke-width="1.2"/><text x="1117" y="181" font-size="17" font-weight="700" fill="#5A6F77" text-anchor="middle" xml:space="preserve">Tools</text>
<rect x="1165" y="158" width="114" height="34" rx="17.0" fill="#F0F3F6" stroke="#C6DCEA" stroke-width="1.2"/><text x="1222" y="181" font-size="17" font-weight="700" fill="#5A6F77" text-anchor="middle" xml:space="preserve">Resources</text>
<rect x="1289" y="158" width="95" height="34" rx="17.0" fill="#F0F3F6" stroke="#C6DCEA" stroke-width="1.2"/><text x="1337" y="181" font-size="17" font-weight="700" fill="#5A6F77" text-anchor="middle" xml:space="preserve">Prompts</text>
<path d="M768,175 H872" fill="none" stroke="#618794" stroke-width="2.4" marker-start="url(#mg-as)" marker-end="url(#mg-ar)"/>
<text x="819" y="161" font-size="15" font-weight="700" fill="#5A6F77" text-anchor="middle" letter-spacing="1.4" xml:space="preserve">MCP</text>
<rect x="880" y="284" width="520" height="82" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5"/>
<text x="900" y="331" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Postgres</text>
<rect x="1079" y="308" width="76" height="34" rx="17.0" fill="#F0F3F6" stroke="#C6DCEA" stroke-width="1.2"/><text x="1117" y="331" font-size="17" font-weight="700" fill="#5A6F77" text-anchor="middle" xml:space="preserve">Tools</text>
<rect x="1165" y="308" width="114" height="34" rx="17.0" fill="#F0F3F6" stroke="#C6DCEA" stroke-width="1.2"/><text x="1222" y="331" font-size="17" font-weight="700" fill="#5A6F77" text-anchor="middle" xml:space="preserve">Resources</text>
<rect x="1289" y="308" width="95" height="34" rx="17.0" fill="#F0F3F6" stroke="#C6DCEA" stroke-width="1.2"/><text x="1337" y="331" font-size="17" font-weight="700" fill="#5A6F77" text-anchor="middle" xml:space="preserve">Prompts</text>
<path d="M768,325 H872" fill="none" stroke="#618794" stroke-width="2.4" marker-start="url(#mg-as)" marker-end="url(#mg-ar)"/>
<text x="819" y="311" font-size="15" font-weight="700" fill="#5A6F77" text-anchor="middle" letter-spacing="1.4" xml:space="preserve">MCP</text>
<text x="880" y="80" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">MCP SERVERS</text>
</svg>
</div>

| Participant | Role |
|---|---|
| **MCP Host** | The AI application (e.g., a Databricks App) that manages one or more MCP clients |
| **MCP Client** | A component inside the host that maintains a dedicated connection to a single MCP server |
| **MCP Server** | A program that exposes tools, resources, or prompts to connected clients |

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">MCP servers expose three types of primitives: <strong>Tools</strong> (executable functions), <strong>Resources</strong> (read-only data sources), and <strong>Prompts</strong> (reusable interaction templates). In practice, tool primitives are the most common integration point for agents.</p>
</div>
</div>
</div>

Model Context Protocol overview (<a href="https://docs.databricks.com/aws/en/agents/mcp-tools" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/agents/mcp-tools/" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/agents/mcp-tools" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

### A2. MCP on Databricks

On Databricks, an MCP is a **Unity Catalog object** with a three-level name, such as `system.ai.github`. Agents call MCPs through **Unity Gateway**, which checks the caller's Unity Catalog permissions: `EXECUTE` on the MCP, plus `USE CATALOG` and `USE SCHEMA` on its catalog and schema. Each MCP still has its own URL, so the agent uses one MCP client per MCP.

<div style="background: #fff; max-width: 1600px; margin: 0 auto; padding: 12px 0 24px;">
<svg viewBox="0 0 1600 744" width="100%" role="img" aria-label="MCP on Databricks. Your app&#x27;s agent uses one MCP client per MCP. Calls to Unity Catalog MCPs go through Unity Gateway, where the caller needs EXECUTE plus USE CATALOG and USE SCHEMA. The MCPs are Unity Catalog objects with full names: Databricks-provided, such as system.ai.github; external, a third-party server you register; and custom, your Databricks Apps server, registered. For your own UC functions, new Python agents call the function directly with UCFunctionToolkit, without MCP, which is recommended; the legacy UC functions MCP server, a workspace MCP endpoint, is the older path." style="display: block; height: auto; font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;">
<defs>
<marker id="md-ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
<filter id="md-sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="3" stdDeviation="6" flood-color="#1B3139" flood-opacity="0.08"/></filter>
</defs>
<defs><marker id="md-as" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker></defs>
<rect x="880" y="20" width="680" height="436" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#md-sh)"/>
<rect x="902" y="42" width="142" height="26" rx="13.0" fill="#FF5F46"/><text x="973" y="60" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">UNITY CATALOG</text>
<text x="902" y="108" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">MCPs</text>
<rect x="902" y="130" width="636" height="92" rx="10" fill="#fff" stroke="#2272B4" stroke-width="1.5"/>
<rect x="922" y="146" width="197" height="26" rx="13.0" fill="#DCEBF8"/><text x="1020" y="164" font-size="13" font-weight="700" fill="#2272B4" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">DATABRICKS-PROVIDED</text>
<text x="1514" y="165" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="end" xml:space="preserve">Hosted by Databricks</text>
<text x="922" y="204" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">system.ai.github</text>
<rect x="902" y="236" width="636" height="92" rx="10" fill="#fff" stroke="#2272B4" stroke-width="1.5"/>
<rect x="922" y="252" width="97" height="26" rx="13.0" fill="#DCEBF8"/><text x="970" y="270" font-size="13" font-weight="700" fill="#2272B4" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">EXTERNAL</text>
<text x="1514" y="271" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="end" xml:space="preserve">A third-party server you register</text>
<text x="922" y="310" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">catalog.schema.partner_mcp</text>
<rect x="902" y="342" width="636" height="92" rx="10" fill="#fff" stroke="#2272B4" stroke-width="1.5"/>
<rect x="922" y="358" width="79" height="26" rx="13.0" fill="#DCEBF8"/><text x="961" y="376" font-size="13" font-weight="700" fill="#2272B4" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">CUSTOM</text>
<text x="1514" y="377" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="end" xml:space="preserve">Your Databricks Apps server, registered</text>
<text x="922" y="416" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">catalog.schema.my_mcp</text>
<rect x="40" y="202.0" width="360" height="160" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#md-sh)"/>
<rect x="62" y="224" width="97" height="26" rx="13.0" fill="#FF5F46"/><text x="110" y="242" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">YOUR APP</text>
<text x="62" y="290" font-size="21" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">Agent</text>
<text x="62" y="324" font-size="19" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">MCP clients, one per MCP</text>
<rect x="480" y="202.0" width="300" height="160" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#md-sh)"/>
<rect x="502" y="224" width="142" height="26" rx="13.0" fill="#FF5F46"/><text x="573" y="242" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">UNITY GATEWAY</text>
<rect x="502" y="270" width="124" height="30" rx="15.0" fill="#E7EBEE"/><text x="564" y="290" font-size="13" font-weight="700" fill="#5A6F77" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">EXECUTE</text>
<text x="502" y="330" font-size="17" font-weight="600" fill="#5A6F77" text-anchor="start" xml:space="preserve">+ USE CATALOG, USE SCHEMA</text>
<path d="M410,282.0 H472" fill="none" stroke="#618794" stroke-width="2.4" marker-start="url(#md-as)" marker-end="url(#md-ar)"/>
<path d="M790,282.0 H830" fill="none" stroke="#618794" stroke-width="2.4" marker-start="url(#md-as)"/>
<path d="M830,282.0 V176.0 H894" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#md-ar)"/>
<path d="M830,282.0 H894" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#md-ar)"/>
<path d="M830,282.0 V388.0 H894" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#md-ar)"/>
<text x="480" y="502" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">YOUR UC FUNCTIONS</text>
<rect x="480" y="520" width="560" height="92" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5"/>
<rect x="502" y="536" width="124" height="26" rx="13.0" fill="#00A972"/><text x="564" y="554" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">RECOMMENDED</text>
<text x="1016" y="555" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="end" xml:space="preserve">New Python agents, no MCP</text>
<text x="502" y="594" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">UCFunctionToolkit</text>
<rect x="480" y="628" width="560" height="92" rx="12" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" stroke-dasharray="7 5"/>
<rect x="502" y="644" width="79" height="26" rx="13.0" fill="#5A6F77"/><text x="541" y="662" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">LEGACY</text>
<text x="1016" y="663" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="end" xml:space="preserve">Workspace MCP endpoint</text>
<text x="502" y="702" font-size="19" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">UC functions MCP server</text>
<rect x="1180" y="520" width="380" height="200" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#md-sh)"/>
<rect x="1202" y="534" width="336" height="5" rx="2.5" fill="#1B5162"/>
<rect x="1202" y="553" width="124" height="26" rx="13.0" fill="#1B5162"/><text x="1264" y="571" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">UC FUNCTION</text>
<text x="1370" y="654" font-size="20" font-weight="700" fill="#0b2026" text-anchor="middle" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">catalog.schema.my_function</text>
<path d="M1050,566.0 H1172" fill="none" stroke="#618794" stroke-width="2.4" marker-start="url(#md-as)" marker-end="url(#md-ar)"/>
<path d="M1050,674.0 H1172" fill="none" stroke="#618794" stroke-width="2.4" stroke-dasharray="7 6" marker-start="url(#md-as)" marker-end="url(#md-ar)"/>
<path d="M220.0,372.0 V566.0 H472" fill="none" stroke="#618794" stroke-width="2.4" marker-start="url(#md-as)" marker-end="url(#md-ar)"/>
<path d="M220.0,566.0 V674.0 H472" fill="none" stroke="#618794" stroke-width="2.4" stroke-dasharray="7 6" marker-end="url(#md-ar)"/>
</svg>
</div>

| Type | Description | Example |
|---|---|---|
| **Databricks-provided** | MCPs in `system.ai` that Databricks hosts, so there is no server to run | `system.ai.dbsql`, `system.ai.genie_one_mcp`, `system.ai.github` |
| **External** | A self-hosted or third-party MCP server you register | A partner's MCP server |
| **Custom** | Your own MCP server, built on Databricks Apps and registered | Domain-specific tools, or a REST API exposed as MCP tools |

**Your own UC functions.** For Unity Catalog functions you register yourself, new Python agents should call them with `UCFunctionToolkit`, which runs registered functions directly instead of through MCP. The older path, the **UC functions MCP server**, is one of the workspace MCP endpoints that Databricks now labels legacy.

MCPs on Databricks (<a href="https://docs.databricks.com/aws/en/agents/mcp-tools" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/agents/mcp-tools/" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/agents/mcp-tools" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

Unity Catalog functions MCP server, legacy (<a href="https://docs.databricks.com/aws/en/agents/mcp-tools/uc-functions" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/agents/mcp-tools/uc-functions/" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/agents/mcp-tools/uc-functions" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

<div style="width: 100%; font-family: sans-serif;">

<div style="margin: 0 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 16px 18px 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07);">

  <div style="height: 5px; border-radius: 5px; margin: -4px 0 12px; background: #FF5F46;"></div>

  <img src="https://files.training.databricks.com/binder/prod_main/deploying-and-monitoring-agents-on-databricks-en_us-1.1.0/images/20261029T133314Z/Deploying and Monitoring Agents on Databricks/Includes/images/genie-code.png" style="height: 64px; margin-bottom: 10px;">

  <div style="font-size: 15pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">
    Want to learn more about <strong>MCP server types on Databricks</strong>? Ask Genie Code. Click on the genie icon <img src="https://files.training.databricks.com/binder/prod_main/deploying-and-monitoring-agents-on-databricks-en_us-1.1.0/images/20261029T133314Z/Deploying and Monitoring Agents on Databricks/Includes/images/genie-icon.png" style="height: 32px; vertical-align: middle;"> and begin querying. For example, click the <strong>Copy</strong> button below and paste into <strong>Genie Code</strong>.
  </div>

  <div style="display: flex; align-items: center; gap: 10px; background: #F9F7F4; border: 1px solid #ddd; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0b2026;">
    <span id="genie-query" style="flex: 1;">What Databricks-provided MCPs are available? How do I connect my agent to the Genie One MCP, system.ai.genie_one_mcp?</span>
    <button onclick="
      var text = document.getElementById('genie-query').innerText;
      var ta = document.createElement('textarea');
      ta.value = text;
      ta.style.position = 'fixed';
      ta.style.opacity = '0';
      document.body.appendChild(ta);
      ta.select();
      document.execCommand('copy');
      document.body.removeChild(ta);
      this.innerText = 'Copied!';
      var btn = this;
      setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
    " style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 4px 12px; font-size: 13pt; cursor: pointer; white-space: nowrap;">Copy</button>
  </div>

</div>

</div>

### A3. MCP Lifecycle

As of protocol version **2026-07-28**, MCP is **stateless**. There is no handshake to open a session and no message to close one. Each request stands alone and carries everything the server needs.

The diagram shows this stateless flow. Databricks MCPs still begin with the earlier `initialize` handshake, described below the diagram.

<div style="background: #fff; max-width: 1600px; margin: 0 auto; padding: 12px 0 24px;">
<svg viewBox="0 0 1600 692" width="100%" role="img" aria-label="How MCP requests work in protocol version 2026-07-28, where each request stands alone. Between an MCP client and the Databricks-provided MCP system.ai.dbsql: 1, list tools: the client sends tools/list, with an optional cursor, and gets execute_sql and poll_sql_result; 2, call a tool: the client sends tools/call for execute_sql with a SQL query for the average price in the Mission and gets content and structured content where avg_price is 188.63. There is no handshake, no session, and no shutdown." style="display: block; height: auto; font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;">
<defs>
<marker id="mr-ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
<filter id="mr-sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="3" stdDeviation="6" flood-color="#1B3139" flood-opacity="0.08"/></filter>
</defs>
<rect x="370" y="20" width="340" height="96" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#mr-sh)"/>
<text x="540" y="62" font-size="21" font-weight="700" fill="#0b2026" text-anchor="middle" xml:space="preserve">MCP client</text>
<text x="540" y="92" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="middle" xml:space="preserve">Inside the host</text>
<rect x="1210" y="20" width="340" height="96" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#mr-sh)"/>
<text x="1380" y="62" font-size="21" font-weight="700" fill="#0b2026" text-anchor="middle" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">system.ai.dbsql</text>
<text x="1380" y="92" font-size="17" font-weight="500" fill="#5A6F77" text-anchor="middle" xml:space="preserve">Databricks-provided MCP</text>
<path d="M546,206 H1370" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#mr-ar)"/>
<text x="960" y="190" font-size="21" font-weight="700" fill="#0b2026" text-anchor="middle" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">tools/list</text>
<text x="960" y="240" font-size="19" font-weight="500" fill="#5A6F77" text-anchor="middle" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">cursor (optional)</text>
<path d="M1374,298 H550" fill="none" stroke="#618794" stroke-width="2.4" stroke-dasharray="7 6" marker-end="url(#mr-ar)"/>
<text x="960" y="282" font-size="21" font-weight="700" fill="#0b2026" text-anchor="middle" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">result</text>
<text x="960" y="332" font-size="19" font-weight="500" fill="#5A6F77" text-anchor="middle" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">execute_sql · poll_sql_result</text>
<path d="M546,442 H1370" fill="none" stroke="#618794" stroke-width="2.4" marker-end="url(#mr-ar)"/>
<text x="960" y="426" font-size="21" font-weight="700" fill="#0b2026" text-anchor="middle" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">tools/call</text>
<text x="960" y="476" font-size="19" font-weight="500" fill="#5A6F77" text-anchor="middle" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">execute_sql {"query": "SELECT AVG(price) ... 'Mission'"}</text>
<path d="M1374,534 H550" fill="none" stroke="#618794" stroke-width="2.4" stroke-dasharray="7 6" marker-end="url(#mr-ar)"/>
<text x="960" y="518" font-size="21" font-weight="700" fill="#0b2026" text-anchor="middle" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">result</text>
<text x="960" y="568" font-size="19" font-weight="500" fill="#5A6F77" text-anchor="middle" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">content + structuredContent: avg_price = 188.63</text>
<text x="960" y="664" font-size="19" font-weight="600" fill="#5A6F77" text-anchor="middle" xml:space="preserve">No handshake · no session · no shutdown</text>
<line x1="540" y1="122" x2="540" y2="622" stroke="#C6DCEA" stroke-width="2" stroke-dasharray="6 6"/>
<line x1="1380" y1="122" x2="1380" y2="622" stroke="#C6DCEA" stroke-width="2" stroke-dasharray="6 6"/>
<circle cx="64" cy="200" r="16" fill="#FF5F46"/><text x="64" y="206" font-size="16" font-weight="800" fill="#fff" text-anchor="middle" xml:space="preserve">1</text>
<text x="92" y="206" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">LIST TOOLS</text>
<line x1="40" y1="392" x2="1560" y2="392" stroke="#EEEDE9" stroke-width="2"/>
<circle cx="64" cy="436" r="16" fill="#FF5F46"/><text x="64" y="442" font-size="16" font-weight="800" fill="#fff" text-anchor="middle" xml:space="preserve">2</text>
<text x="92" y="442" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">CALL A TOOL</text>
</svg>
</div>

A client works with a server in a few kinds of requests:
- **Discover (optional).** `server/discover` returns the versions, capabilities, and identity a server supports in one call. A client can skip it and send requests directly.
- **List and call tools.** `tools/list` returns each tool's name, description, and input schema, a page at a time. `tools/call` runs one tool and returns a `content` array, often with `structuredContent`.
- **Handle a version mismatch.** If the server doesn't support the requested version, it returns an `UnsupportedProtocolVersionError` that lists the versions it does support, and the client retries with one of them.

Three details worth knowing:
- **The host chooses the tool.** MCP gives each tool's name, description, and input schema. Deciding when to call one is up to the host application and its model, not the protocol.
- **Errors come back two ways.** If the tool fails or its arguments are invalid, the server returns a normal result with `isError: true`, so the model can correct itself and retry. An unknown tool, a malformed request, or a server error returns a JSON-RPC error.
- **Earlier versions used a handshake.** Versions up to `2025-11-25` open a session with an `initialize` request first. Databricks' MCPs still use this handshake today, and the MCP client in the example below performs it for you.

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: Connecting an agent to an MCP</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;"># Connect an agent to a Databricks-provided MCP
from agents import Agent, Runner, set_default_openai_api, set_default_openai_client, set_trace_processors
from databricks.sdk import WorkspaceClient
from databricks_openai import AsyncDatabricksOpenAI
from databricks_openai.agents import McpServer
<span></span>
# Send the Agents SDK's model calls through Unity Gateway
set_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))
set_default_openai_api("responses")
set_trace_processors([])  # Turn off the SDK's own tracing, which sends traces to OpenAI
<span></span>
workspace_client = WorkspaceClient()
<span></span>
async with McpServer(
    url=f"{workspace_client.config.host}/ai-gateway/mcp-services/system.ai.dbsql",
    workspace_client=workspace_client,
    name="dbsql",
) as dbsql:
    # The MCP client sends tools/list and tools/call for the agent as needed
    agent = Agent(
        name="my_agent",
        model="my_catalog.my_schema.agent-model",
        mcp_servers=[dbsql],
    )
    result = await Runner.run(agent, "What is the average listing price in the Mission?")</div>
</div>
</details>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">Tools are discovered at runtime, not hardcoded. When a server adds a tool, connected agents pick it up the next time they list tools, with no redeployment. When MLflow Tracing is active, every MCP tool call is captured as a span within the trace.</p>
</div>
</div>
</div>

MCP versioning and compatibility, 2026-07-28 (<a href="https://modelcontextprotocol.io/specification/2026-07-28/basic/versioning" target="_blank" style="color: #1976d2; text-decoration: underline;">modelcontextprotocol.io</a>)

MCP tools, 2026-07-28 (<a href="https://modelcontextprotocol.io/specification/2026-07-28/server/tools" target="_blank" style="color: #1976d2; text-decoration: underline;">modelcontextprotocol.io</a>)

### A4. MCP in a Deployed App

In a Databricks App, open the MCP connection inside each request, so concurrent requests never share one connection. The app's service principal needs `EXECUTE` on each MCP it calls, plus access to the data its tools touch, such as `SELECT` on the tables a SQL query reads.

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: FastAPI server with MCP tool integration</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;"># server.py -- FastAPI app with MCP integration
import os
<span></span>
import mlflow
from agents import Agent, Runner, set_default_openai_api, set_default_openai_client, set_trace_processors
from databricks.sdk import WorkspaceClient
from databricks_openai import AsyncDatabricksOpenAI
from databricks_openai.agents import McpServer
from fastapi import FastAPI
from pydantic import BaseModel
<span></span>
# Send model calls through Unity Gateway and trace them in MLflow
set_default_openai_client(AsyncDatabricksOpenAI(use_ai_gateway=True))
set_default_openai_api("responses")
set_trace_processors([])
mlflow.openai.autolog()
<span></span>
workspace_client = WorkspaceClient()
app = FastAPI()
<span></span>
<span></span>
class ChatRequest(BaseModel):
    message: str
<span></span>
<span></span>
@app.post("/chat")
async def chat(request: ChatRequest):
    # Open a connection to the MCP for each request
    async with McpServer(
        url=f"{workspace_client.config.host}/ai-gateway/mcp-services/system.ai.dbsql",
        workspace_client=workspace_client,
        name="dbsql",
    ) as dbsql:
        agent = Agent(
            name="deployed_agent",
            model=os.environ["MODEL_SERVICE"],
            mcp_servers=[dbsql],
        )
        result = await Runner.run(agent, request.message)
    return {"response": result.final_output}</div>
</div>
</details>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">The model service is declared as a resource in <code>databricks.yml</code>, and <code>app.yaml</code> passes its name to the app as <code>MODEL_SERVICE</code> with <code>valueFrom</code>, keeping the app code portable across environments. Opening <code>McpServer</code> with <code>async with</code> inside the request closes the connection when the request finishes.</p>
</div>
</div>
</div>


## B. Observability with MLflow Tracing

### B1. Adding Tracing to Your Agent

MLflow Tracing records the execution of each agent request as a structured **trace**, a tree of **spans** that represent individual operations. The simplest way to enable tracing is **automatic tracing**: one call to `mlflow.openai.autolog()` traces the agent's model and tool calls through the OpenAI SDK, including the OpenAI Agents SDK.

Each span captures:
- **Inputs and outputs**: What went into the operation and what came out
- **Timing**: Start time, end time, and duration
- **Span type**: The kind of operation (e.g., `AGENT`, `CHAT_MODEL`, `TOOL`, `RETRIEVER`)
- **Status**: Whether the operation succeeded or failed

For your own logic, such as the agent's entry point or a preprocessing step, add **custom spans** with the `@mlflow.trace` decorator or `mlflow.start_span()`. They join the same trace as the automatic spans.

<div style="background: #fff; max-width: 1600px; margin: 0 auto; padding: 12px 0 24px;">
<svg viewBox="0 0 1600 680" width="100%" role="img" aria-label="An autolog trace from an agent app as a span waterfall over about 13 seconds: the AgentRunner.run AGENT span contains three Response CHAT_MODEL spans for the model calls and two execute_sql TOOL spans, one that looks at the table and one that computes the average. The selected execute_sql span shows its input, a SQL query, its output, SUCCEEDED with 188.63, a latency of 0.88 seconds, and that it ran through system.ai.dbsql." style="display: block; height: auto; font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Arial,sans-serif;">
<defs>
<marker id="tw-ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="8" markerHeight="8" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
<filter id="tw-sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="3" stdDeviation="6" flood-color="#1B3139" flood-opacity="0.08"/></filter>
</defs>
<rect x="40" y="20" width="1060" height="640" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#tw-sh)"/>
<text x="68" y="62" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">TRACE · AGENT APP</text>
<text x="68" y="100" font-size="24" font-weight="700" fill="#0b2026" text-anchor="start" xml:space="preserve">What is the average listing price in the Mission?</text>
<text x="68" y="152" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">SPAN</text>
<text x="440" y="152" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">TYPE</text>
<text x="620" y="152" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">TIMELINE</text>
<line x1="620" y1="166" x2="620" y2="540" stroke="#EEEDE9" stroke-width="1.5"/>
<text x="620" y="566" font-size="17" font-weight="500" fill="#618794" text-anchor="middle" xml:space="preserve">0 s</text>
<line x1="760" y1="166" x2="760" y2="540" stroke="#EEEDE9" stroke-width="1.5"/>
<text x="760" y="566" font-size="17" font-weight="500" fill="#618794" text-anchor="middle" xml:space="preserve">5 s</text>
<line x1="900" y1="166" x2="900" y2="540" stroke="#EEEDE9" stroke-width="1.5"/>
<text x="900" y="566" font-size="17" font-weight="500" fill="#618794" text-anchor="middle" xml:space="preserve">10 s</text>
<text x="72" y="208" font-size="18" font-weight="600" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">AgentRunner.run</text>
<rect x="440" y="188" width="70" height="26" rx="13.0" fill="#1B5162"/><text x="475" y="206" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">AGENT</text>
<rect x="620" y="189" width="353" height="24" rx="6" fill="#1B5162" fill-opacity="0.85"/>
<text x="983" y="208" font-size="17" font-weight="500" fill="#618794" text-anchor="start" xml:space="preserve">12.59 s</text>
<path d="M84,226 V262 H96" fill="none" stroke="#C6DCEA" stroke-width="1.5"/>
<text x="102" y="268" font-size="18" font-weight="600" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">Response</text>
<rect x="440" y="248" width="115" height="26" rx="13.0" fill="#2272B4"/><text x="498" y="266" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">CHAT_MODEL</text>
<rect x="650" y="249" width="98" height="24" rx="6" fill="#2272B4" fill-opacity="0.85"/>
<text x="757" y="268" font-size="17" font-weight="500" fill="#618794" text-anchor="start" xml:space="preserve">3.49 s</text>
<path d="M84,286 V322 H96" fill="none" stroke="#C6DCEA" stroke-width="1.5"/>
<text x="102" y="328" font-size="18" font-weight="600" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">execute_sql</text>
<rect x="440" y="308" width="60" height="26" rx="13.0" fill="#00A972"/><text x="470" y="326" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">TOOL</text>
<rect x="747" y="309" width="52" height="24" rx="6" fill="#00A972" fill-opacity="0.85"/>
<text x="809" y="328" font-size="17" font-weight="500" fill="#618794" text-anchor="start" xml:space="preserve">1.85 s</text>
<path d="M84,346 V382 H96" fill="none" stroke="#C6DCEA" stroke-width="1.5"/>
<text x="102" y="388" font-size="18" font-weight="600" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">Response</text>
<rect x="440" y="368" width="115" height="26" rx="13.0" fill="#2272B4"/><text x="498" y="386" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">CHAT_MODEL</text>
<rect x="813" y="369" width="58" height="24" rx="6" fill="#2272B4" fill-opacity="0.85"/>
<text x="880" y="388" font-size="17" font-weight="500" fill="#618794" text-anchor="start" xml:space="preserve">2.06 s</text>
<rect x="56" y="414" width="1030" height="54" rx="10" fill="#FFF6F4" stroke="#FABFBA"/>
<path d="M84,406 V442 H96" fill="none" stroke="#C6DCEA" stroke-width="1.5"/>
<text x="102" y="448" font-size="18" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">execute_sql</text>
<rect x="440" y="428" width="60" height="26" rx="13.0" fill="#00A972"/><text x="470" y="446" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">TOOL</text>
<rect x="870" y="429" width="25" height="24" rx="6" fill="#00A972" fill-opacity="1"/>
<text x="905" y="448" font-size="17" font-weight="500" fill="#618794" text-anchor="start" xml:space="preserve">0.88 s</text>
<path d="M84,466 V502 H96" fill="none" stroke="#C6DCEA" stroke-width="1.5"/>
<text x="102" y="508" font-size="18" font-weight="600" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">Response</text>
<rect x="440" y="488" width="115" height="26" rx="13.0" fill="#2272B4"/><text x="498" y="506" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">CHAT_MODEL</text>
<rect x="905" y="489" width="68" height="24" rx="6" fill="#2272B4" fill-opacity="0.85"/>
<text x="983" y="508" font-size="17" font-weight="500" fill="#618794" text-anchor="start" xml:space="preserve">2.42 s</text>
<text x="68" y="630" font-size="17" font-weight="400" fill="#5A6F77" text-anchor="start" xml:space="preserve">Autolog adds the model and tool spans. Spans nest like a call tree.</text>
<rect x="1130" y="20" width="430" height="640" rx="14" fill="#fff" stroke="#C6DCEA" stroke-width="1.5" filter="url(#tw-sh)"/>
<rect x="1152" y="34" width="386" height="5" rx="2.5" fill="#FF5F46"/>
<rect x="1152" y="53" width="142" height="26" rx="13.0" fill="#FF5F46"/><text x="1223" y="71" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">SELECTED SPAN</text>
<text x="1152" y="120" font-size="22" font-weight="700" fill="#0b2026" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">execute_sql</text>
<rect x="1152" y="136" width="60" height="26" rx="13.0" fill="#00A972"/><text x="1182" y="154" font-size="13" font-weight="700" fill="#fff" text-anchor="middle" letter-spacing="0.6" xml:space="preserve">TOOL</text>
<text x="1152" y="202" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">INPUTS</text>
<rect x="1152" y="214" width="386" height="50" rx="8" fill="#303F47"/>
<text x="1168" y="246" font-size="16" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">query: <tspan style="font-family: inherit;" fill="#FFCC66">"SELECT AVG(price) ..."</tspan></text>
<text x="1152" y="300" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">OUTPUTS</text>
<rect x="1152" y="312" width="386" height="50" rx="8" fill="#303F47"/>
<text x="1168" y="344" font-size="16" font-weight="400" fill="#E6EBEE" text-anchor="start" style="font-family: ui-monospace,Menlo,Consolas,monospace;" xml:space="preserve">SUCCEEDED · 188.63</text>
<text x="1152" y="398" font-size="15" font-weight="700" fill="#618794" text-anchor="start" letter-spacing="1.4" xml:space="preserve">ATTRIBUTES</text>
<text x="1152" y="430" font-size="17" font-weight="500" fill="#618794" text-anchor="start" xml:space="preserve">latency</text>
<text x="1538" y="430" font-size="17" font-weight="600" fill="#0b2026" text-anchor="end" xml:space="preserve">0.88 s</text>
<text x="1152" y="462" font-size="17" font-weight="500" fill="#618794" text-anchor="start" xml:space="preserve">status</text>
<text x="1538" y="462" font-size="17" font-weight="600" fill="#0b2026" text-anchor="end" xml:space="preserve">OK</text>
<text x="1152" y="494" font-size="17" font-weight="500" fill="#618794" text-anchor="start" xml:space="preserve">via</text>
<text x="1538" y="494" font-size="17" font-weight="600" fill="#0b2026" text-anchor="end" xml:space="preserve">system.ai.dbsql</text>
</svg>
</div>

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: Automatic tracing and custom spans</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">import mlflow
<span></span>
# Automatic tracing: one line traces every model and tool call the agent makes
mlflow.openai.autolog()
<span></span>
<span></span>
# Custom spans: trace your own logic in the same trace
@mlflow.trace(span_type="AGENT")
def handle_request(user_message: str) -&amp;gt; str:
    with mlflow.start_span(name="preprocessing") as span:
        cleaned = preprocess(user_message)
        span.set_inputs({"raw": user_message})
        span.set_outputs({"cleaned": cleaned})
    return agent.run(cleaned)</div>
</div>
</details>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">A typical agent trace contains a root <code>AGENT</code> span with child spans for <code>CHAT_MODEL</code> calls, <code>TOOL</code> invocations, and <code>RETRIEVER</code> operations. This hierarchy shows exactly how the agent processed the request: which tools it called, in what order, and how long each step took.</p>
</div>
</div>
</div>

Automatic tracing for the OpenAI Agents SDK (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/tracing/integrations/openai-agent" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/tracing/integrations/openai-agent" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/tracing/integrations/openai-agent" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

Manual tracing (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/tracing/manual-tracing" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/tracing/manual-tracing" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/tracing/manual-tracing" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

### B2. Traces in Production

In production, traces are logged to an **MLflow Experiment**, the central hub where observability and evaluation data come together. You can also store the experiment's traces in Unity Catalog, as B3 shows.

Traces are the foundation that the rest of the monitoring pipeline builds on:
- **Scorers evaluate traces**, not raw requests. Any agent that produces MLflow traces can be evaluated using the same scorer infrastructure.
- **Unity Catalog storage** keeps traces in Delta tables for SQL analysis and dashboards.
- **Metric backfill** applies new scorers retroactively to traces logged before the scorer existed.

A Databricks App gets its experiment through an experiment resource declared in `databricks.yml`. `app.yaml` passes the experiment's ID to the app as `MLFLOW_EXPERIMENT_ID` with `valueFrom`, and the agent sets it at startup:

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: Setting the MLflow experiment at startup</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">import mlflow
import os
<span></span>
mlflow.set_experiment(experiment_id=os.environ["MLFLOW_EXPERIMENT_ID"])</div>
</div>
</details>

**Tracing in production.** Three settings matter once real traffic arrives:

| Setting | What it does |
|---|---|
| `mlflow-tracing` package | A smaller package with only what tracing needs. Install it in place of the full `mlflow` package, not alongside it. |
| `MLFLOW_ENABLE_ASYNC_TRACE_LOGGING` | Logs traces in the background so requests don't wait. On by default outside notebooks. |
| `MLFLOW_TRACE_SAMPLING_RATIO` | Exports only a share of traces, such as `0.1` for 10%. The default, `1.0`, keeps every trace. |

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">Tracing must be enabled before scorers can be useful. If your agent is not producing traces, there is nothing for scorers to evaluate. The first step in any monitoring setup is ensuring traces are flowing to an MLflow Experiment.</p>
</div>
</div>
</div>

MLflow Tracing overview (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/tracing/overview" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/tracing/overview" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/tracing/overview" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

Production tracing (<a href="https://mlflow.org/docs/latest/genai/tracing/prod-tracing" target="_blank" style="color: #1976d2; text-decoration: underline;">mlflow.org</a>)

### B3. Storing Traces in Unity Catalog

Databricks recommends storing traces for new and production workloads in **Unity Catalog Delta tables**, in OpenTelemetry format. This gives you long-term retention with no per-experiment trace limit, SQL-queryable trace data, and governance through UC schema and table permissions rather than experiment-level ACLs. Without a Unity Catalog location, traces fall back to the experiment's own store, which holds up to 100,000 traces per experiment.

Choose Unity Catalog storage when you create the experiment, by passing `trace_location`. You can't add it to an existing experiment. Writing traces also needs a SQL warehouse, set with `MLFLOW_TRACING_SQL_WAREHOUSE_ID`, and MLflow 3.14 or later:

<details style="margin: 14px 4px; background: #fff; border: 1.5px solid #C6DCEA; border-radius: 14px; padding: 0 18px; box-shadow: 0 2px 8px rgba(27,49,57,.07); font-family: sans-serif;">
<summary style="cursor: pointer; padding: 16px 0; font-weight: 700; font-size: 18px; color: #0b2026;">Example: Creating a UC-backed experiment</summary>
<div class="code-card" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
<div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
  <div style="display:inline-flex;align-items:center;gap:0.45rem;">
    <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
    <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
  </div>
  <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=b.closest('.code-card').querySelector('.code-card-body').textContent,t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
    <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
    <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
    <span class="copy-label">Copy</span>
  </button>
</div>
<div class="code-card-body" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;white-space:pre-wrap;word-break:break-word;">import os
<span></span>
import mlflow
from mlflow.entities.trace_location import UnityCatalog
<span></span>
os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = "&amp;lt;SQL_WAREHOUSE_ID&amp;gt;"
<span></span>
experiment = mlflow.set_experiment(
    experiment_name="my-agent-experiment",
    trace_location=UnityCatalog(
        catalog_name="prod",
        schema_name="agents",
        table_prefix="agent",
    ),
)</div>
</div>
</details>

This automatically creates four Delta tables in the specified schema:

| Table | Contents |
|---|---|
| `&lt;prefix&gt;_otel_spans` | Individual span records (the primary trace data) |
| `&lt;prefix&gt;_otel_annotations` | Assessments and feedback attached to traces |
| `&lt;prefix&gt;_otel_logs` | Log entries associated with trace execution |
| `&lt;prefix&gt;_otel_metrics` | OpenTelemetry metric data points recorded alongside traces |

<div class="mlt">
<style>
  .mlt{background:#ffffff;max-width:1600px;margin:0 auto;padding:24px 20px 40px;}
  .mlt svg{display:block;width:100%;height:auto;}
  .mlt svg text{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Arial,sans-serif;fill:#0b2026;}
  .mlt .h   {font-weight:800;}
  .mlt .b   {font-weight:700;}
  .mlt .m   {fill:#618794;font-weight:500;}
  .mlt .mono,.mlt .mono text{white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace;font-variant-ligatures:none;font-feature-settings:"liga" 0,"calt" 0;}
  .mlt .pfx {fill:#FF5F46;}
</style>
<svg viewBox="0 0 1600 680" role="img" aria-label="MLflow traces registered with Unity Catalog. Your agent app, multi-turn sessions, tool calls, evaluation runs, autologged frameworks, or custom code are instrumented by MLflow Tracing, which captures inputs and outputs, latency, token usage, tool calls, errors and span hierarchy. Binding the experiment to UnityCatalog with table_prefix agent writes traces into the prod.agents schema as four Delta tables: agent_otel_spans, agent_otel_logs, agent_otel_annotations and agent_otel_metrics, plus two SQL views built over them: agent_trace_unified and agent_trace_metadata. The assets are governed by Unity Catalog and queried with Databricks SQL or mlflow.search_traces.">
  <defs>
    <!-- repo glyphs: product-icons/tableIcon.svg and product-icons/TableViewIcon.svg -->
    <symbol id="ic-table" viewBox="0 0 16 16">
      <path fill-rule="evenodd" clip-rule="evenodd" d="M1 1.75C1 1.33579 1.33579 1 1.75 1H14.25C14.6642 1 15 1.33579 15 1.75V14.25C15 14.6642 14.6642 15 14.25 15H1.75C1.33579 15 1 14.6642 1 14.25V1.75ZM2.5 2.5V5.5H13.5V2.5H2.5ZM2.5 13.5V7H5V13.5H2.5ZM6.5 13.5H9.5V7H6.5V13.5ZM11 7V13.5H13.5V7H11Z" fill="#0B2026"/>
    </symbol>
    <symbol id="ic-view" viewBox="0 0 16 16">
      <path fill-rule="evenodd" clip-rule="evenodd" d="M1.75 1C1.33579 1 1 1.33579 1 1.75V14.25C1 14.6642 1.33579 15 1.75 15H4V13.5H2.5V7H5V9H6.5V7H9.5V9H11V7H13.5V9H15V1.75C15 1.33579 14.6642 1 14.25 1H1.75ZM13.5 5.5V2.5H2.5V5.5H13.5Z" fill="#FF3621"/>
      <path fill-rule="evenodd" clip-rule="evenodd" d="M11.75 10C11.5511 10 11.3603 10.079 11.2197 10.2197C11.1398 10.2995 11.0798 10.3955 11.0429 10.5H9.95735C9.85439 10.2087 9.57658 10 9.25003 10L5.75 10C5.55108 10 5.36031 10.079 5.21966 10.2197C5.07901 10.3603 4.99999 10.5511 5 10.75L5.00003 12.5C5.00003 13.8807 6.11932 15 7.50003 15C8.88074 15 10 13.8807 10 12.5V12H11L11 12.5C11 13.8807 12.1193 15 13.5 15C14.8807 15 16 13.8807 16 12.5V10.7506C16 10.3364 15.6642 10 15.25 10L11.75 10ZM12.5 12.5L12.5 11.5L14.5 11.5V12.5C14.5 13.0523 14.0523 13.5 13.5 13.5C12.9477 13.5 12.5 13.0523 12.5 12.5ZM6.50002 11.5L6.50003 12.5C6.50003 13.0523 6.94774 13.5 7.50003 13.5C8.05231 13.5 8.50003 13.0523 8.50003 12.5V11.5001L6.50002 11.5Z" fill="#FF3621"/>
    </symbol>
    <marker id="ar" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#618794"/></marker>
    <marker id="ar-big" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="6" markerHeight="6" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#1B5162"/></marker>
    <filter id="sh" x="-10%" y="-10%" width="120%" height="125%"><feDropShadow dx="0" dy="4" stdDeviation="9" flood-color="#1B3139" flood-opacity="0.10"/></filter>
  </defs>
  <!-- ================= 1 · GenAI app ================= -->
  <rect x="40" y="20" width="320" height="620" rx="18" fill="#ffffff" stroke="#DCE0E2" stroke-width="1.5" filter="url(#sh)"/>
  <text class="m b" x="68" y="60" font-size="16" letter-spacing="1.6">1 · YOUR AGENT</text>
  <text class="h" x="68" y="90" font-size="25">Anything that traces</text>
  <text class="m" x="68" y="116" font-size="17">the course agent, or any framework</text>
  <g>
    <rect x="62" y="142" width="276" height="62" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
    <text class="b" x="82" y="168" font-size="19">Your agent app</text>
    <text class="mono m" x="82" y="190" font-size="15">@mlflow.trace</text>
    <rect x="62" y="216" width="276" height="62" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
    <text class="b" x="82" y="242" font-size="19">Multi-turn sessions</text>
    <text class="mono m" x="82" y="264" font-size="15">update_current_trace()</text>
    <rect x="62" y="290" width="276" height="62" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
    <text class="b" x="82" y="316" font-size="19">Tool calls</text>
    <text class="mono m" x="82" y="338" font-size="15">mlflow.start_span()</text>
    <rect x="62" y="364" width="276" height="62" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
    <text class="b" x="82" y="390" font-size="19">Evaluation runs</text>
    <text class="mono m" x="82" y="412" font-size="15">mlflow.genai.evaluate()</text>
    <rect x="62" y="438" width="276" height="62" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
    <text class="b" x="82" y="464" font-size="19">OpenAI, LangChain, DSPy</text>
    <text class="mono m" x="82" y="486" font-size="15">mlflow.&amp;lt;flavor&amp;gt;.autolog()</text>
    <rect x="62" y="512" width="276" height="62" rx="10" fill="#F9F7F4" stroke="#EEEDE9"/>
    <text class="b" x="82" y="538" font-size="19">Custom code</text>
    <text class="mono m" x="82" y="560" font-size="15">@mlflow.trace</text>
  </g>
  <text class="m" x="200" y="612" font-size="16" text-anchor="middle">no changes to app logic</text>
  <!-- arrow 1 -> 2 -->
  <path d="M362,330 L414,330" fill="none" stroke="#618794" stroke-width="2.5" marker-end="url(#ar)"/>
  <!-- ================= 2 · MLflow Tracing ================= -->
  <rect x="420" y="20" width="320" height="620" rx="18" fill="#ffffff" stroke="#DCE0E2" stroke-width="1.5" filter="url(#sh)"/>
  <text class="m b" x="448" y="60" font-size="16" letter-spacing="1.6">2 · MLFLOW TRACING</text>
  <text class="h" x="448" y="90" font-size="25">Each trace captures</text>
  <g font-size="19">
    <circle cx="456" cy="130" r="4" fill="#4299E0"/><text x="470" y="136">Inputs &amp; outputs</text>
    <circle cx="456" cy="162" r="4" fill="#4299E0"/><text x="470" y="168">Latency per step</text>
    <circle cx="456" cy="194" r="4" fill="#4299E0"/><text x="470" y="200">Token usage</text>
    <circle cx="456" cy="226" r="4" fill="#4299E0"/><text x="470" y="232">Tool / function calls</text>
    <circle cx="456" cy="258" r="4" fill="#4299E0"/><text x="470" y="264">Errors &amp; retries</text>
    <circle cx="456" cy="290" r="4" fill="#4299E0"/><text x="470" y="296">Span hierarchy</text>
  </g>
  <text class="m b" x="448" y="350" font-size="16" letter-spacing="1.6">BIND TO UNITY CATALOG</text>
  <rect x="440" y="364" width="280" height="168" rx="10" fill="#303F47"/>
  <g class="mono" font-size="14">
    <text x="456" y="392" style="fill:#C4CCD6">mlflow.set_experiment(</text>
    <text x="456" y="414" style="fill:#C4CCD6">  experiment_name="…",</text>
    <text x="456" y="436" style="fill:#C4CCD6">  trace_location=UnityCatalog(</text>
    <text x="456" y="458" style="fill:#C4CCD6">    catalog_name=<tspan style="font-family: inherit; fill:#FFCC66">"prod"</tspan>,</text>
    <text x="456" y="480" style="fill:#C4CCD6">    schema_name=<tspan style="font-family: inherit; fill:#FFCC66">"agents"</tspan>,</text>
    <text x="456" y="502" style="fill:#C4CCD6">    table_prefix=<tspan style="font-family: inherit; fill:#FF8774">"agent"</tspan>))</text>
  </g>
  <text class="m" x="448" y="570" font-size="16">Needs a SQL warehouse:</text>
  <text class="mono m" x="448" y="592" font-size="15">MLFLOW_TRACING_SQL_WAREHOUSE_ID</text>
  <!-- arrow 2 -> 3 -->
  <path d="M742,330 L794,330" fill="none" stroke="#618794" stroke-width="2.5" marker-end="url(#ar)"/>
  <!-- ================= 3 · Unity Catalog ================= -->
  <rect x="800" y="20" width="760" height="620" rx="18" fill="#ffffff" stroke="#1B5162" stroke-width="1.8" filter="url(#sh)"/>
  <path d="M818,20 H1542 A18,18 0 0 1 1560,38 V76 H800 V38 A18,18 0 0 1 818,20 Z" fill="#1B5162"/>
  <text x="826" y="56" font-size="25" font-weight="800" style="fill:#ffffff">3 · Unity Catalog</text>
  <text class="mono" x="1534" y="55" font-size="16" text-anchor="end" style="fill:#ffffff">prod.agents.<tspan style="font-family: inherit; fill:#FF8774">agent</tspan>_*</text>
  <!-- tables section -->
  <use href="#ic-table" x="826" y="96" width="20" height="20"/>
  <text class="b" x="854" y="113" font-size="19">4 Delta tables</text>
  <text class="m" x="982" y="113" font-size="17">raw OpenTelemetry data, written as traces arrive</text>
  <g>
    <!-- spans -->
    <path d="M822,130 H1164 A10,10 0 0 1 1174,140 V212 A10,10 0 0 1 1164,222 H822 Z" fill="#F9F7F4" stroke="#EEEDE9"/>
    <rect x="822" y="130" width="5" height="92" fill="#1B5162"/>
    <use href="#ic-table" x="842" y="146" width="22" height="22"/>
    <text class="mono b" x="874" y="164" font-size="18"><tspan style="font-family: inherit;" class="pfx">agent</tspan>_otel_spans</text>
    <text class="m" x="842" y="192" font-size="17">One row per span: the call tree with</text>
    <text class="m" x="842" y="211" font-size="17">timing, inputs and outputs</text>
    <!-- logs -->
    <path d="M1186,130 H1528 A10,10 0 0 1 1538,140 V212 A10,10 0 0 1 1528,222 H1186 Z" fill="#F9F7F4" stroke="#EEEDE9"/>
    <rect x="1186" y="130" width="5" height="92" fill="#1B5162"/>
    <use href="#ic-table" x="1206" y="146" width="22" height="22"/>
    <text class="mono b" x="1238" y="164" font-size="18"><tspan style="font-family: inherit;" class="pfx">agent</tspan>_otel_logs</text>
    <text class="m" x="1206" y="192" font-size="17">OpenTelemetry log records emitted</text>
    <text class="m" x="1206" y="211" font-size="17">while the app runs</text>
    <!-- annotations -->
    <path d="M822,234 H1164 A10,10 0 0 1 1174,244 V316 A10,10 0 0 1 1164,326 H822 Z" fill="#F9F7F4" stroke="#EEEDE9"/>
    <rect x="822" y="234" width="5" height="92" fill="#1B5162"/>
    <use href="#ic-table" x="842" y="250" width="22" height="22"/>
    <text class="mono b" x="874" y="268" font-size="18"><tspan style="font-family: inherit;" class="pfx">agent</tspan>_otel_annotations</text>
    <text class="m" x="842" y="296" font-size="17">Assessments, feedback and tags</text>
    <text class="m" x="842" y="315" font-size="17">attached to traces</text>
    <!-- metrics -->
    <path d="M1186,234 H1528 A10,10 0 0 1 1538,244 V316 A10,10 0 0 1 1528,326 H1186 Z" fill="#F9F7F4" stroke="#EEEDE9"/>
    <rect x="1186" y="234" width="5" height="92" fill="#1B5162"/>
    <use href="#ic-table" x="1206" y="250" width="22" height="22"/>
    <text class="mono b" x="1238" y="268" font-size="18"><tspan style="font-family: inherit;" class="pfx">agent</tspan>_otel_metrics</text>
    <text class="m" x="1206" y="296" font-size="17">OpenTelemetry metric data points</text>
    <text class="m" x="1206" y="315" font-size="17">recorded alongside traces</text>
  </g>
  <!-- bus: tables -> views -->
  <g fill="none" stroke="#1B5162" stroke-width="2.2" stroke-linecap="round">
    <path d="M998,326 L998,364"/>
    <path d="M1362,326 L1362,364"/>
    <path d="M998,364 L1362,364"/>
    <path d="M998,364 L998,406" marker-end="url(#ar-big)"/>
    <path d="M1362,364 L1362,406" marker-end="url(#ar-big)"/>
  </g>
  <rect x="1056" y="350" width="252" height="28" rx="14" fill="#ffffff" stroke="#1B5162" stroke-width="1.5"/>
  <use href="#ic-view" x="1068" y="356" width="16" height="16"/>
  <text class="b" x="1196" y="370" font-size="16" text-anchor="middle">2 SQL views over the tables</text>
  <!-- views -->
  <g>
    <path d="M822,412 H1164 A10,10 0 0 1 1174,422 V526 A10,10 0 0 1 1164,536 H822 Z" fill="#FFF6F4" stroke="#FABFBA"/>
    <rect x="822" y="412" width="5" height="124" fill="#FF5F46"/>
    <use href="#ic-view" x="842" y="428" width="22" height="22"/>
    <text class="mono b" x="874" y="446" font-size="18"><tspan style="font-family: inherit;" class="pfx">agent</tspan>_trace_unified</text>
    <text class="m" x="842" y="476" font-size="17">One row per trace: spans, metadata</text>
    <text class="m" x="842" y="495" font-size="17">and assessments together</text>
    <rect x="842" y="506" width="120" height="24" rx="12" fill="#FF5F46"/>
    <text x="902" y="523" font-size="14" font-weight="700" text-anchor="middle" style="fill:#ffffff">FULL DETAIL</text>
    <path d="M1186,412 H1528 A10,10 0 0 1 1538,422 V526 A10,10 0 0 1 1528,536 H1186 Z" fill="#FFF6F4" stroke="#FABFBA"/>
    <rect x="1186" y="412" width="5" height="124" fill="#FF5F46"/>
    <use href="#ic-view" x="1206" y="428" width="22" height="22"/>
    <text class="mono b" x="1238" y="446" font-size="18"><tspan style="font-family: inherit;" class="pfx">agent</tspan>_trace_metadata</text>
    <text class="m" x="1206" y="476" font-size="17">Tags, metadata and assessments only,</text>
    <text class="m" x="1206" y="495" font-size="17">no span payloads</text>
    <rect x="1206" y="506" width="156" height="24" rx="12" fill="#FF5F46"/>
    <text x="1284" y="523" font-size="14" font-weight="700" text-anchor="middle" style="fill:#ffffff">FAST FILTERING</text>
  </g>
  <!-- what you get -->
  <line x1="822" y1="556" x2="1538" y2="556" stroke="#EEEDE9" stroke-width="1.5"/>
  <g font-size="16">
    <rect x="822" y="574" width="236" height="44" rx="22" fill="#F9F7F4" stroke="#EEEDE9"/>
    <text x="940" y="601" text-anchor="middle"><tspan style="font-family: inherit;" class="b">Govern</tspan><tspan style="font-family: inherit;" class="m">: grants · lineage</tspan></text>
    <rect x="1070" y="574" width="212" height="44" rx="22" fill="#F9F7F4" stroke="#EEEDE9"/>
    <text x="1176" y="601" text-anchor="middle"><tspan style="font-family: inherit;" class="b">Query</tspan><tspan style="font-family: inherit;" class="m">: Databricks SQL</tspan></text>
    <rect x="1294" y="574" width="244" height="44" rx="22" fill="#F9F7F4" stroke="#EEEDE9"/>
    <text x="1416" y="601" text-anchor="middle"><tspan style="font-family: inherit;" class="b">Read</tspan><tspan style="font-family: inherit;" class="m">: </tspan><tspan style="font-family: inherit;" class="mono m" font-size="14">mlflow.search_traces()</tspan></text>
  </g>
</svg>
</div>

Once stored in Delta tables, traces can be queried directly with SQL through a Databricks SQL warehouse, enabling custom reporting and dashboards.

**Governance**: Access is controlled through UC table permissions. Users need `USE CATALOG`, `USE SCHEMA`, and explicit `MODIFY` + `SELECT` grants on each table. Note that `ALL_PRIVILEGES` is **not sufficient** — `MODIFY` and `SELECT` must be granted explicitly.

**Monitoring**: Scorers that run in production read UC traces through a SQL warehouse saved on the experiment with `mlflow.tracing.set_databricks_monitoring_sql_warehouse_id()`. The `MLFLOW_TRACING_SQL_WAREHOUSE_ID` environment variable only applies to your own process.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">Once you bind an experiment to a UC trace location, you <strong>cannot reassign</strong> it to a different location. However, multiple experiments can share the same UC location. Users with access to the UC tables can view all traces stored there, regardless of which experiment the traces belong to. Ingestion is capped at 200 traces per second per workspace and 100 MB per second per table.</p>
</div>
</div>
</div>

Traces in Unity Catalog (<a href="https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog" target="_blank" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/mlflow3/genai/tracing/trace-unity-catalog" target="_blank" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/mlflow3/genai/tracing/trace-unity-catalog" target="_blank" style="color: #1976d2; text-decoration: underline;">GCP</a>)

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>

